# Performance Analysis Dashboard

Interactive analysis and visualization of extraction model performance over time.

This notebook provides:
- **Trend Analysis**: Track improvements/regressions over time
- **Plan Comparison**: Compare different extraction strategies
- **Cost Analysis**: Monitor spending and efficiency
- **Benchmark Evaluation**: Compare against performance targets
- **Statistical Insights**: Advanced analysis and patterns

## Data Source
Performance data is automatically recorded from test runs to `history.json` via the PerformanceTracker.


## Section 1: Setup and Data Loading


In [ ]:
# Imports and Configuration
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent.parent))

# Core data analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Performance tracking
from tests.performance.tracker import PerformanceTracker
from tests.performance.benchmarks import (
    PERFORMANCE_TARGETS,
    COST_TARGETS,
    QUALITY_TARGETS,
    evaluate_performance
)
from tests.performance.utils import (
    load_performance_data,
    detect_regressions,
    calculate_improvements,
    generate_insights
)

# Utilities
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✅ Imports and configuration complete")


In [ ]:
# Load Performance History
tracker = PerformanceTracker()
history = tracker.history

print(f"📊 Loaded {len(history)} test runs from history.json")

if history:
    first_run = min(history, key=lambda x: x['timestamp'])
    last_run = max(history, key=lambda x: x['timestamp'])
    print(f"📅 Date range: {first_run['timestamp'][:10]} to {last_run['timestamp'][:10]}")
    
    # Get unique plans tested
    all_plans = set()
    for run in history:
        all_plans.update(run.get('results', {}).keys())
    print(f"🔬 Plans tested: {', '.join(sorted(all_plans))}")
else:
    print("⚠️  No test runs recorded yet. Run tests to generate data.")


In [ ]:
# Data Preprocessing - Convert to DataFrames
data = load_performance_data()

runs_df = data['runs']
plans_df = data['plans']
types_df = data['types']

print("📋 DataFrames created:")
print(f"  - Runs DataFrame: {len(runs_df)} rows")
print(f"  - Plans DataFrame: {len(plans_df)} rows")
print(f"  - Types DataFrame: {len(types_df)} rows")

if not plans_df.empty:
    print("\n📊 Plans DataFrame preview:")
    display(plans_df.head())
else:
    print("\n⚠️  No data available. Run tests first to generate performance data.")


## Section 2: Overview Dashboard


In [ ]:
# Summary Statistics
if not plans_df.empty:
    summary_stats = {
        'Total Runs': len(runs_df),
        'Date Range': f"{runs_df['timestamp'].min().date()} to {runs_df['timestamp'].max().date()}" if not runs_df.empty else "N/A",
        'Plans Tested': plans_df['plan_name'].nunique(),
        'Total Cost': f"${plans_df['total_cost'].sum():.4f}",
        'Average Confidence (All Plans)': f"{plans_df['average_confidence'].mean():.2f}",
    }
    
    print("📊 Summary Statistics")
    print("=" * 50)
    for key, value in summary_stats.items():
        print(f"{key:<30}: {value}")
    
    # Best performance per plan
    print("\n🏆 Best Performance per Plan:")
    print("-" * 50)
    best_per_plan = plans_df.loc[plans_df.groupby('plan_name')['average_confidence'].idxmax()]
    for _, row in best_per_plan.iterrows():
        print(f"{row['plan_name']:<15} | Confidence: {row['average_confidence']:.2f} | "
              f"Cost: ${row['total_cost']:.4f} | Run: {row['run_id']}")
else:
    print("⚠️  No data available for summary statistics.")


In [ ]:
# Key Metrics Overview
if not plans_df.empty:
    # Latest run
    latest_run_id = runs_df['run_id'].iloc[-1] if not runs_df.empty else None
    latest_plans = plans_df[plans_df['run_id'] == latest_run_id] if latest_run_id else pd.DataFrame()
    
    if not latest_plans.empty:
        print("📈 Latest Run Summary")
        print("=" * 70)
        print(f"Run ID: {latest_run_id}")
        print(f"Timestamp: {latest_plans['timestamp'].iloc[0]}")
        print(f"Notes: {latest_plans['notes'].iloc[0] if latest_plans['notes'].iloc[0] else 'None'}")
        print("\nPlan Performance:")
        print("-" * 70)
        for _, row in latest_plans.iterrows():
            print(f"{row['plan_name']:<15} | Conf: {row['average_confidence']:.2f} | "
                  f"Cost: ${row['total_cost']:.4f} | Time: {row['processing_time']:.2f}s | "
                  f"Errors: {row['validation_errors']}")
    
    # Average metrics
    print("\n📊 Average Metrics (All Runs):")
    print("-" * 70)
    avg_metrics = plans_df.groupby('plan_name').agg({
        'average_confidence': 'mean',
        'total_cost': 'mean',
        'processing_time': 'mean',
        'validation_errors': 'mean'
    }).round(4)
    display(avg_metrics)
else:
    print("⚠️  No data available.")


In [ ]:
# Quick Visualizations
if not plans_df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Average confidence by plan
    avg_conf = plans_df.groupby('plan_name')['average_confidence'].mean().sort_values(ascending=False)
    axes[0, 0].bar(avg_conf.index, avg_conf.values, color='steelblue')
    axes[0, 0].set_title('Average Confidence by Plan (All Runs)', fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('Average Confidence')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].axhline(y=0.8, color='r', linestyle='--', label='Target (0.8)')
    axes[0, 0].legend()
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Average cost by plan
    avg_cost = plans_df.groupby('plan_name')['total_cost'].mean().sort_values(ascending=False)
    axes[0, 1].bar(avg_cost.index, avg_cost.values, color='coral')
    axes[0, 1].set_title('Average Cost by Plan (All Runs)', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Average Cost ($)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Cost vs Confidence scatter
    for plan in plans_df['plan_name'].unique():
        plan_data = plans_df[plans_df['plan_name'] == plan]
        axes[1, 0].scatter(plan_data['total_cost'], plan_data['average_confidence'], 
                          label=plan, alpha=0.6, s=100)
    axes[1, 0].set_title('Cost vs Confidence (All Runs)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Total Cost ($)')
    axes[1, 0].set_ylabel('Average Confidence')
    axes[1, 0].axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Timeline - Number of runs over time
    if not runs_df.empty:
        runs_df['date'] = pd.to_datetime(runs_df['timestamp']).dt.date
        run_counts = runs_df.groupby('date').size()
        axes[1, 1].plot(run_counts.index, run_counts.values, marker='o', linewidth=2, markersize=8)
        axes[1, 1].set_title('Test Runs Over Time', fontsize=12, fontweight='bold')
        axes[1, 1].set_xlabel('Date')
        axes[1, 1].set_ylabel('Number of Runs')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available for visualization.")


## Section 3: Trend Analysis


In [ ]:
# Confidence Score Trends Over Time
if not plans_df.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Plot each plan
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
        
        if not plan_data.empty:
            # Line plot
            ax.plot(plan_data['timestamp'], plan_data['average_confidence'], 
                   marker='o', label=plan_name, linewidth=2, markersize=6, alpha=0.7)
            
            # Moving average (window of 2 if enough data)
            if len(plan_data) >= 2:
                window = min(2, len(plan_data))
                plan_data['ma'] = plan_data['average_confidence'].rolling(window=window, center=True).mean()
                ax.plot(plan_data['timestamp'], plan_data['ma'], 
                       linestyle='--', alpha=0.5, linewidth=1)
            
            # Annotate best and worst
            best_idx = plan_data['average_confidence'].idxmax()
            worst_idx = plan_data['average_confidence'].idxmin()
            if best_idx != worst_idx:
                best_row = plan_data.loc[best_idx]
                worst_row = plan_data.loc[worst_idx]
                ax.annotate(f"Best: {best_row['average_confidence']:.2f}", 
                          xy=(best_row['timestamp'], best_row['average_confidence']),
                          xytext=(10, 10), textcoords='offset points', fontsize=8,
                          bbox=dict(boxstyle='round,pad=0.3', facecolor='green', alpha=0.3))
    
    ax.axhline(y=0.8, color='r', linestyle='--', linewidth=2, label='Target (0.8)', alpha=0.7)
    ax.set_title('Confidence Score Trends Over Time', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Timestamp', fontsize=12)
    ax.set_ylabel('Average Confidence', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available for trend analysis.")


In [ ]:
# Cost Trends Over Time
if not plans_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Cost over time
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
        if not plan_data.empty:
            axes[0].plot(plan_data['timestamp'], plan_data['total_cost'], 
                        marker='o', label=plan_name, linewidth=2, markersize=6, alpha=0.7)
    
    axes[0].set_title('Cost Trends Over Time', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Timestamp')
    axes[0].set_ylabel('Total Cost ($)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    # Cumulative cost
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
        if not plan_data.empty:
            plan_data['cumulative_cost'] = plan_data['total_cost'].cumsum()
            axes[1].plot(plan_data['timestamp'], plan_data['cumulative_cost'], 
                        marker='o', label=plan_name, linewidth=2, markersize=6, alpha=0.7)
    
    axes[1].set_title('Cumulative Cost Over Time', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Timestamp')
    axes[1].set_ylabel('Cumulative Cost ($)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Cost efficiency (confidence/cost ratio)
    if 'cost_per_confidence' in plans_df.columns:
        print("\n💰 Cost Efficiency (Confidence per Dollar):")
        print("-" * 50)
        efficiency = plans_df.groupby('plan_name').agg({
            'cost_per_confidence': 'mean'
        }).sort_values('cost_per_confidence')
        efficiency['efficiency_score'] = 1 / efficiency['cost_per_confidence']
        display(efficiency)
else:
    print("⚠️  No data available.")


In [ ]:
# Processing Time and Validation Error Trends
if not plans_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Processing time trends
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
        if not plan_data.empty:
            axes[0].plot(plan_data['timestamp'], plan_data['processing_time'], 
                        marker='o', label=plan_name, linewidth=2, markersize=6, alpha=0.7)
    
    axes[0].axhline(y=60, color='r', linestyle='--', label='Target (60s)', alpha=0.7)
    axes[0].set_title('Processing Time Trends', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Timestamp')
    axes[0].set_ylabel('Processing Time (seconds)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    # Validation errors
    error_data = plans_df.groupby(['timestamp', 'plan_name'])['validation_errors'].sum().reset_index()
    if not error_data.empty:
        for plan_name in sorted(error_data['plan_name'].unique()):
            plan_errors = error_data[error_data['plan_name'] == plan_name].sort_values('timestamp')
            axes[1].bar(range(len(plan_errors)), plan_errors['validation_errors'], 
                       label=plan_name, alpha=0.7)
        axes[1].set_title('Validation Errors per Run', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Run Index')
        axes[1].set_ylabel('Number of Errors')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available.")


## Section 4: Plan Comparison


In [ ]:
# Side-by-Side Plan Comparison
if not plans_df.empty:
    # Latest run comparison
    latest_run_id = plans_df['run_id'].iloc[-1] if not plans_df.empty else None
    if latest_run_id:
        latest_comparison = plans_df[plans_df['run_id'] == latest_run_id][
            ['plan_name', 'average_confidence', 'total_cost', 'processing_time', 'validation_errors']
        ].set_index('plan_name')
        
        print("📊 Latest Run Comparison:")
        print("=" * 70)
        display(latest_comparison)
    
    # Best run comparison
    best_runs = plans_df.loc[plans_df.groupby('plan_name')['average_confidence'].idxmax()]
    best_comparison = best_runs[
        ['plan_name', 'average_confidence', 'total_cost', 'processing_time', 'validation_errors', 'run_id']
    ].set_index('plan_name')
    
    print("\n🏆 Best Run Comparison (All Time):")
    print("=" * 70)
    display(best_comparison)
    
    # Average performance comparison
    avg_comparison = plans_df.groupby('plan_name').agg({
        'average_confidence': 'mean',
        'total_cost': 'mean',
        'processing_time': 'mean',
        'validation_errors': 'mean'
    }).round(4)
    
    print("\n📈 Average Performance Comparison (All Runs):")
    print("=" * 70)
    display(avg_comparison)
else:
    print("⚠️  No data available.")


In [ ]:
# Quality vs Cost Analysis with Pareto Frontier
if not plans_df.empty:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Color map for plans
    plan_colors = {plan: plt.cm.tab10(i) for i, plan in enumerate(sorted(plans_df['plan_name'].unique()))}
    
    # Scatter plot
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name]
        ax.scatter(plan_data['total_cost'], plan_data['average_confidence'], 
                  label=plan_name, alpha=0.6, s=100, color=plan_colors[plan_name])
    
    # Target lines
    ax.axhline(y=0.8, color='r', linestyle='--', linewidth=2, label='Confidence Target (0.8)', alpha=0.7)
    ax.axvline(x=0.10, color='orange', linestyle='--', linewidth=2, label='Cost Target ($0.10)', alpha=0.7)
    
    # Highlight Pareto frontier (best quality/cost trade-offs)
    # Simple approach: find points with best confidence/cost ratio
    plans_df['efficiency'] = plans_df['average_confidence'] / (plans_df['total_cost'] + 0.0001)
    top_efficient = plans_df.nlargest(5, 'efficiency')
    ax.scatter(top_efficient['total_cost'], top_efficient['average_confidence'], 
              s=200, marker='*', color='gold', edgecolors='black', linewidth=2, 
              label='Top 5 Efficient', zorder=5)
    
    ax.set_title('Quality vs Cost Analysis', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Total Cost ($)', fontsize=12)
    ax.set_ylabel('Average Confidence', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Interpretation:")
    print("  - Points in top-right quadrant: High quality, high cost")
    print("  - Points in bottom-left quadrant: Low quality, low cost")
    print("  - Gold stars: Most cost-efficient (best quality/cost ratio)")
    print("  - Ideal: High confidence, low cost (top-left area)")
else:
    print("⚠️  No data available.")


In [ ]:
# Extraction Type Breakdown
if not types_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Stacked bar chart: Confidence by extraction type per plan
    pivot_data = types_df.pivot_table(
        values='confidence', 
        index='plan_name', 
        columns='extract_type', 
        aggfunc='mean'
    )
    
    pivot_data.plot(kind='bar', stacked=False, ax=axes[0], width=0.8)
    axes[0].set_title('Average Confidence by Extraction Type', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Confidence Score')
    axes[0].set_xlabel('Plan')
    axes[0].legend(title='Extraction Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[0].axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Heatmap: Plans × Extraction Types
    sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='YlOrRd', 
                vmin=0, vmax=1, ax=axes[1], cbar_kws={'label': 'Confidence Score'})
    axes[1].set_title('Confidence Heatmap (Plans × Extraction Types)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Extraction Type')
    axes[1].set_ylabel('Plan')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Extraction Type Performance Summary:")
    print("-" * 70)
    display(pivot_data)
else:
    print("⚠️  No data available for extraction type analysis.")


## Section 5: Run-by-Run Analysis


In [ ]:
# Run Selector - Display available runs
if not runs_df.empty:
    print("📋 Available Test Runs:")
    print("=" * 70)
    for idx, row in runs_df.iterrows():
        print(f"{idx + 1}. Run ID: {row['run_id']} | Date: {row['timestamp']} | Notes: {row.get('notes', 'None')}")
    
    # Select a run to analyze (change this index to analyze different runs)
    selected_run_idx = len(runs_df) - 1  # Default to latest run
    selected_run = runs_df.iloc[selected_run_idx]
    selected_run_id = selected_run['run_id']
    
    print(f"\n🔍 Selected Run: {selected_run_id}")
    print(f"   Timestamp: {selected_run['timestamp']}")
    print(f"   Notes: {selected_run.get('notes', 'None')}")
    
    # Display run details
    run_plans = plans_df[plans_df['run_id'] == selected_run_id]
    if not run_plans.empty:
        print("\n📊 Plan Performance for Selected Run:")
        print("-" * 70)
        display(run_plans[['plan_name', 'average_confidence', 'total_cost', 
                           'processing_time', 'validation_errors']].set_index('plan_name'))
else:
    print("⚠️  No runs available.")
    selected_run_id = None


In [ ]:
# Detailed Run Analysis
if selected_run_id and not types_df.empty:
    run_types = types_df[types_df['run_id'] == selected_run_id]
    
    if not run_types.empty:
        print(f"📊 Extraction Type Breakdown for Run {selected_run_id}:")
        print("=" * 70)
        
        # Pivot table: Plans × Extraction Types
        type_pivot = run_types.pivot_table(
            values='confidence',
            index='plan_name',
            columns='extract_type',
            aggfunc='mean'
        )
        display(type_pivot)
        
        # Visualization
        fig, ax = plt.subplots(figsize=(12, 6))
        type_pivot.plot(kind='bar', ax=ax, width=0.8)
        ax.set_title(f'Confidence by Extraction Type - Run {selected_run_id}', 
                    fontsize=12, fontweight='bold')
        ax.set_ylabel('Confidence Score')
        ax.set_xlabel('Plan')
        ax.legend(title='Extraction Type', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  No extraction type data available for this run.")
else:
    print("⚠️  No run selected or no data available.")


In [ ]:
# Compare Two Runs
if len(runs_df) >= 2:
    # Select two runs to compare (default: first and last)
    run1_id = runs_df.iloc[0]['run_id']
    run2_id = runs_df.iloc[-1]['run_id']
    
    print(f"🔄 Comparing Runs:")
    print(f"   Run 1: {run1_id} ({runs_df.iloc[0]['timestamp']})")
    print(f"   Run 2: {run2_id} ({runs_df.iloc[-1]['timestamp']})")
    print("=" * 70)
    
    comparison = tracker.compare_runs(run1_id, run2_id)
    
    if 'error' not in comparison:
        # Create comparison DataFrame
        comp_data = []
        for plan_name, changes in comparison['plans'].items():
            comp_data.append({
                'Plan': plan_name,
                'Confidence Change': f"{changes['confidence_change']:+.3f}",
                'Cost Change': f"${changes['cost_change']:+.4f}",
                'Time Change': f"{changes['time_change']:+.2f}s",
                'Errors Change': f"{changes['errors_change']:+d}"
            })
        
        comp_df = pd.DataFrame(comp_data)
        display(comp_df)
        
        # Visualization
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        run1_data = plans_df[plans_df['run_id'] == run1_id]
        run2_data = plans_df[plans_df['run_id'] == run2_id]
        
        # Confidence comparison
        merged = run1_data.merge(run2_data, on='plan_name', suffixes=('_run1', '_run2'))
        x = np.arange(len(merged))
        width = 0.35
        axes[0, 0].bar(x - width/2, merged['average_confidence_run1'], width, label=f'Run 1 ({run1_id[:8]})', alpha=0.8)
        axes[0, 0].bar(x + width/2, merged['average_confidence_run2'], width, label=f'Run 2 ({run2_id[:8]})', alpha=0.8)
        axes[0, 0].set_ylabel('Confidence')
        axes[0, 0].set_title('Confidence Comparison')
        axes[0, 0].set_xticks(x)
        axes[0, 0].set_xticklabels(merged['plan_name'], rotation=45)
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3, axis='y')
        
        # Cost comparison
        axes[0, 1].bar(x - width/2, merged['total_cost_run1'], width, label=f'Run 1', alpha=0.8)
        axes[0, 1].bar(x + width/2, merged['total_cost_run2'], width, label=f'Run 2', alpha=0.8)
        axes[0, 1].set_ylabel('Cost ($)')
        axes[0, 1].set_title('Cost Comparison')
        axes[0, 1].set_xticks(x)
        axes[0, 1].set_xticklabels(merged['plan_name'], rotation=45)
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3, axis='y')
        
        # Time comparison
        axes[1, 0].bar(x - width/2, merged['processing_time_run1'], width, label=f'Run 1', alpha=0.8)
        axes[1, 0].bar(x + width/2, merged['processing_time_run2'], width, label=f'Run 2', alpha=0.8)
        axes[1, 0].set_ylabel('Time (s)')
        axes[1, 0].set_title('Processing Time Comparison')
        axes[1, 0].set_xticks(x)
        axes[1, 0].set_xticklabels(merged['plan_name'], rotation=45)
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3, axis='y')
        
        # Errors comparison
        axes[1, 1].bar(x - width/2, merged['validation_errors_run1'], width, label=f'Run 1', alpha=0.8)
        axes[1, 1].bar(x + width/2, merged['validation_errors_run2'], width, label=f'Run 2', alpha=0.8)
        axes[1, 1].set_ylabel('Errors')
        axes[1, 1].set_title('Validation Errors Comparison')
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(merged['plan_name'], rotation=45)
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️  Error: {comparison.get('error', 'Unknown error')}")
else:
    print("⚠️  Need at least 2 runs to compare.")


## Section 6: Statistical Analysis


In [ ]:
# Performance Distribution Analysis
if not plans_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Histogram: Confidence distribution per plan
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name]
        axes[0].hist(plan_data['average_confidence'], alpha=0.6, label=plan_name, bins=15)
    axes[0].axvline(x=0.8, color='r', linestyle='--', linewidth=2, label='Target (0.8)', alpha=0.7)
    axes[0].set_title('Confidence Score Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Average Confidence')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Box plots: Confidence distribution per plan
    plan_list = [plans_df[plans_df['plan_name'] == plan]['average_confidence'].values 
                 for plan in sorted(plans_df['plan_name'].unique())]
    axes[1].boxplot(plan_list, labels=sorted(plans_df['plan_name'].unique()))
    axes[1].axhline(y=0.8, color='r', linestyle='--', linewidth=2, label='Target (0.8)', alpha=0.7)
    axes[1].set_title('Confidence Distribution (Box Plot)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Average Confidence')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Statistical summary
    print("\n📊 Statistical Summary by Plan:")
    print("=" * 70)
    stats_summary = plans_df.groupby('plan_name')['average_confidence'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).round(4)
    display(stats_summary)
else:
    print("⚠️  No data available.")


In [ ]:
# Correlation Analysis
if not plans_df.empty and len(plans_df) > 1:
    # Select numeric columns for correlation
    numeric_cols = ['average_confidence', 'total_cost', 'processing_time', 'validation_errors', 
                    'total_tokens', 'input_tokens', 'output_tokens']
    available_cols = [col for col in numeric_cols if col in plans_df.columns]
    
    if len(available_cols) >= 2:
        corr_matrix = plans_df[available_cols].corr()
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                   square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
        ax.set_title('Correlation Matrix: Performance Metrics', fontsize=12, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
        print("\n💡 Key Correlations:")
        print("-" * 70)
        # Find strongest correlations
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = corr_matrix.iloc[i, j]
                if abs(corr_val) > 0.3:  # Significant correlation
                    print(f"{corr_matrix.columns[i]} ↔ {corr_matrix.columns[j]}: {corr_val:.3f}")
    else:
        print("⚠️  Not enough numeric columns for correlation analysis.")
else:
    print("⚠️  Need at least 2 data points for correlation analysis.")


In [ ]:
# Regression Detection
if not plans_df.empty:
    regressions = detect_regressions(plans_df, threshold=0.05)
    
    if not regressions.empty:
        print("⚠️  Performance Regressions Detected:")
        print("=" * 70)
        display(regressions)
        
        # Visualize regressions
        if len(regressions) > 0:
            fig, ax = plt.subplots(figsize=(12, 6))
            for _, reg in regressions.iterrows():
                plan_data = plans_df[plans_df['plan_name'] == reg['plan_name']].sort_values('timestamp')
                ax.plot(plan_data['timestamp'], plan_data['average_confidence'], 
                       marker='o', label=reg['plan_name'], alpha=0.5)
                # Highlight regression point
                reg_point = plan_data[plan_data['run_id'] == reg['run_id']]
                if not reg_point.empty:
                    ax.scatter(reg_point['timestamp'], reg_point['average_confidence'], 
                             s=200, marker='X', color='red', zorder=5)
            
            ax.set_title('Performance Regressions Highlighted', fontsize=12, fontweight='bold')
            ax.set_xlabel('Timestamp')
            ax.set_ylabel('Average Confidence')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    else:
        print("✅ No significant regressions detected (threshold: 0.05 confidence drop)")
else:
    print("⚠️  No data available.")


In [ ]:
# Improvement Tracking
if not plans_df.empty:
    improvements = calculate_improvements(plans_df, baseline_plan="Baseline")
    
    if not improvements.empty:
        print("📈 Improvement Analysis (vs Baseline):")
        print("=" * 70)
        
        # Summary by plan
        improvement_summary = improvements.groupby('plan_name').agg({
            'confidence_improvement': 'mean',
            'confidence_improvement_pct': 'mean',
            'cost_difference': 'mean'
        }).round(4)
        display(improvement_summary)
        
        # Visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Confidence improvement over time
        for plan_name in sorted(improvements['plan_name'].unique()):
            plan_improvements = improvements[improvements['plan_name'] == plan_name].sort_values('timestamp')
            axes[0].plot(plan_improvements['timestamp'], plan_improvements['confidence_improvement'], 
                        marker='o', label=plan_name, linewidth=2, markersize=6)
        
        axes[0].axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
        axes[0].set_title('Confidence Improvement Over Time (vs Baseline)', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Timestamp')
        axes[0].set_ylabel('Confidence Improvement')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
        
        # Cost difference over time
        for plan_name in sorted(improvements['plan_name'].unique()):
            plan_improvements = improvements[improvements['plan_name'] == plan_name].sort_values('timestamp')
            axes[1].plot(plan_improvements['timestamp'], plan_improvements['cost_difference'], 
                        marker='o', label=plan_name, linewidth=2, markersize=6)
        
        axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
        axes[1].set_title('Cost Difference Over Time (vs Baseline)', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Timestamp')
        axes[1].set_ylabel('Cost Difference ($)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  No baseline data available for comparison.")
else:
    print("⚠️  No data available.")


In [ ]:
# Generate Automated Insights
if not plans_df.empty:
    insights = generate_insights(plans_df, types_df)
    
    print("💡 Automated Insights:")
    print("=" * 70)
    print(f"🏆 Best Overall Plan: {insights.get('best_plan', 'N/A')}")
    print(f"⚠️  Worst Overall Plan: {insights.get('worst_plan', 'N/A')}")
    print(f"💰 Most Cost-Efficient Plan: {insights.get('most_cost_efficient', 'N/A')}")
    print(f"⚡ Fastest Plan: {insights.get('fastest_plan', 'N/A')}")
    print(f"📊 Most Consistent Plan: {insights.get('most_consistent', 'N/A')}")
    
    if insights.get('trends'):
        print("\n📈 Performance Trends:")
        print("-" * 70)
        for trend in insights['trends']:
            trend_icon = "📈" if trend['trend'] == 'improving' else ("📉" if trend['trend'] == 'declining' else "➡️")
            print(f"{trend_icon} {trend['plan']:<15} | {trend['trend'].upper():<10} | Change: {trend['change']:+.3f}")
else:
    print("⚠️  No data available for insights.")


## Section 7: Benchmark Evaluation


In [ ]:
# Benchmark Compliance
if not plans_df.empty:
    # Evaluate each plan against benchmarks
    benchmark_results = []
    
    for plan_name in plans_df['plan_name'].unique():
        plan_data = plans_df[plans_df['plan_name'] == plan_name]
        latest_run = plan_data.iloc[-1]  # Use latest run
        
        # Convert to format expected by evaluate_performance
        plan_result = {
            'confidence_scores': {},  # Would need to get from types_df
            'validation_errors': [None] * int(latest_run['validation_errors']),
            'total_cost': latest_run['total_cost'],
            'processing_time': latest_run['processing_time']
        }
        
        # Get confidence scores from types_df if available
        if not types_df.empty:
            latest_types = types_df[types_df['run_id'] == latest_run['run_id']]
            latest_types_plan = latest_types[latest_types['plan_name'] == plan_name]
            if not latest_types_plan.empty:
                plan_result['confidence_scores'] = dict(zip(
                    latest_types_plan['extract_type'], 
                    latest_types_plan['confidence']
                ))
        
        evaluation = evaluate_performance(plan_result, plan_name)
        benchmark_results.append(evaluation)
    
    benchmark_df = pd.DataFrame(benchmark_results)
    
    print("📊 Benchmark Compliance (Latest Run):")
    print("=" * 70)
    
    # Create visual compliance table
    compliance_cols = ['plan_name', 'meets_confidence_threshold', 'meets_error_threshold', 
                      'meets_cost_target', 'meets_time_target', 'overall_passing']
    display(benchmark_df[compliance_cols].set_index('plan_name'))
    
    # Visualize compliance
    fig, ax = plt.subplots(figsize=(12, 6))
    compliance_data = benchmark_df.set_index('plan_name')[
        ['meets_confidence_threshold', 'meets_error_threshold', 
         'meets_cost_target', 'meets_time_target']
    ].astype(int)
    
    compliance_data.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('Benchmark Compliance by Plan', fontsize=12, fontweight='bold')
    ax.set_ylabel('Pass (1) / Fail (0)')
    ax.set_xlabel('Plan')
    ax.legend(title='Benchmark', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_ylim(0, 1.2)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available.")


In [ ]:
# Target Progress
if not plans_df.empty:
    print("🎯 Progress Toward Targets:")
    print("=" * 70)
    
    target_conf = PERFORMANCE_TARGETS['target_confidence']
    target_cost = PERFORMANCE_TARGETS['max_cost_per_memo']
    
    progress_data = []
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name]
        latest = plan_data.iloc[-1]
        avg_conf = plan_data['average_confidence'].mean()
        
        conf_progress = min(100, (latest['average_confidence'] / target_conf) * 100)
        cost_progress = min(100, (1 - (latest['total_cost'] / target_cost)) * 100) if latest['total_cost'] < target_cost else 0
        
        progress_data.append({
            'Plan': plan_name,
            'Confidence Progress': f"{conf_progress:.1f}%",
            'Cost Progress': f"{cost_progress:.1f}%",
            'Current Confidence': f"{latest['average_confidence']:.2f}",
            'Target Confidence': target_conf,
            'Current Cost': f"${latest['total_cost']:.4f}",
            'Target Cost': f"${target_cost:.2f}"
        })
    
    progress_df = pd.DataFrame(progress_data)
    display(progress_df)
    
    # Visual progress bars
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    plans = progress_df['Plan']
    conf_progress = [float(p.replace('%', '')) for p in progress_df['Confidence Progress']]
    cost_progress = [float(p.replace('%', '')) for p in progress_df['Cost Progress']]
    
    axes[0].barh(plans, conf_progress, color='steelblue', alpha=0.7)
    axes[0].axvline(x=100, color='r', linestyle='--', linewidth=2, label='Target (100%)')
    axes[0].set_title('Progress Toward Confidence Target', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Progress (%)')
    axes[0].set_xlim(0, 120)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='x')
    
    axes[1].barh(plans, cost_progress, color='coral', alpha=0.7)
    axes[1].axvline(x=100, color='r', linestyle='--', linewidth=2, label='Target (100%)')
    axes[1].set_title('Progress Toward Cost Target', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Progress (%)')
    axes[1].set_xlim(0, 120)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available.")


## Section 8: Cost Analysis


In [ ]:
# Cost Breakdown
if not plans_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Pie chart: Total cost by plan
    total_cost_by_plan = plans_df.groupby('plan_name')['total_cost'].sum()
    axes[0].pie(total_cost_by_plan.values, labels=total_cost_by_plan.index, autopct='%1.1f%%',
               startangle=90, colors=plt.cm.Set3.colors)
    axes[0].set_title('Total Cost Distribution by Plan', fontsize=12, fontweight='bold')
    
    # Stacked area chart: Cumulative cost over time
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
        if not plan_data.empty:
            plan_data['cumulative_cost'] = plan_data['total_cost'].cumsum()
            axes[1].fill_between(plan_data['timestamp'], plan_data['cumulative_cost'], 
                                 alpha=0.6, label=plan_name)
    
    axes[1].set_title('Cumulative Cost Over Time', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Timestamp')
    axes[1].set_ylabel('Cumulative Cost ($)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("\n💰 Cost Summary:")
    print("=" * 70)
    cost_summary = plans_df.groupby('plan_name').agg({
        'total_cost': ['sum', 'mean', 'min', 'max']
    }).round(4)
    cost_summary.columns = ['Total Cost', 'Average Cost', 'Min Cost', 'Max Cost']
    display(cost_summary)
else:
    print("⚠️  No data available.")


In [ ]:
# Cost Efficiency Analysis
if not plans_df.empty:
    # Calculate cost per confidence point
    plans_df['cost_per_confidence'] = plans_df.apply(
        lambda x: x['total_cost'] / x['average_confidence'] if x['average_confidence'] > 0 else np.inf,
        axis=1
    )
    plans_df['confidence_per_dollar'] = plans_df.apply(
        lambda x: x['average_confidence'] / (x['total_cost'] + 0.0001),
        axis=1
    )
    
    print("💰 Cost Efficiency Metrics:")
    print("=" * 70)
    efficiency_summary = plans_df.groupby('plan_name').agg({
        'cost_per_confidence': 'mean',
        'confidence_per_dollar': 'mean',
        'average_confidence': 'mean',
        'total_cost': 'mean'
    }).round(4)
    efficiency_summary.columns = ['Cost per Confidence Point', 'Confidence per Dollar', 
                                 'Avg Confidence', 'Avg Cost']
    efficiency_summary = efficiency_summary.sort_values('confidence_per_dollar', ascending=False)
    display(efficiency_summary)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Confidence per dollar
    avg_efficiency = plans_df.groupby('plan_name')['confidence_per_dollar'].mean().sort_values(ascending=False)
    axes[0].bar(avg_efficiency.index, avg_efficiency.values, color='green', alpha=0.7)
    axes[0].set_title('Cost Efficiency: Confidence per Dollar', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Confidence per Dollar')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Cost efficiency trend
    for plan_name in sorted(plans_df['plan_name'].unique()):
        plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
        if not plan_data.empty:
            axes[1].plot(plan_data['timestamp'], plan_data['confidence_per_dollar'], 
                        marker='o', label=plan_name, linewidth=2, markersize=6, alpha=0.7)
    
    axes[1].set_title('Cost Efficiency Trend Over Time', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Timestamp')
    axes[1].set_ylabel('Confidence per Dollar')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No data available.")


## Section 9: Export and Reporting


In [ ]:
# Generate Summary Report
from datetime import datetime

def generate_performance_report(runs_df, plans_df, types_df, output_dir="runtime/outputs/performance_reports"):
    """Generate a markdown performance report"""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_file = output_path / f"performance_report_{timestamp}.md"
    
    lines = []
    lines.append("# Performance Analysis Report")
    lines.append(f"\n**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("")
    lines.append("---")
    lines.append("")
    
    if plans_df.empty:
        lines.append("No performance data available.")
        report_file.write_text("\n".join(lines), encoding="utf-8")
        return str(report_file)
    
    # Summary
    lines.append("## Summary Statistics")
    lines.append("")
    lines.append(f"- **Total Runs:** {len(runs_df)}")
    lines.append(f"- **Date Range:** {runs_df['timestamp'].min().date()} to {runs_df['timestamp'].max().date()}")
    lines.append(f"- **Plans Tested:** {plans_df['plan_name'].nunique()}")
    lines.append(f"- **Total Cost:** ${plans_df['total_cost'].sum():.4f}")
    lines.append("")
    
    # Best performance
    lines.append("## Best Performance by Plan")
    lines.append("")
    best_per_plan = plans_df.loc[plans_df.groupby('plan_name')['average_confidence'].idxmax()]
    for _, row in best_per_plan.iterrows():
        lines.append(f"- **{row['plan_name']}**: Confidence {row['average_confidence']:.2f}, "
                    f"Cost ${row['total_cost']:.4f}, Run {row['run_id']}")
    lines.append("")
    
    # Insights
    insights = generate_insights(plans_df, types_df)
    lines.append("## Key Insights")
    lines.append("")
    lines.append(f"- **Best Overall Plan:** {insights.get('best_plan', 'N/A')}")
    lines.append(f"- **Most Cost-Efficient:** {insights.get('most_cost_efficient', 'N/A')}")
    lines.append(f"- **Fastest Plan:** {insights.get('fastest_plan', 'N/A')}")
    lines.append("")
    
    # Trends
    if insights.get('trends'):
        lines.append("## Performance Trends")
        lines.append("")
        for trend in insights['trends']:
            lines.append(f"- **{trend['plan']}**: {trend['trend'].upper()} (change: {trend['change']:+.3f})")
        lines.append("")
    
    report_file.write_text("\n".join(lines), encoding="utf-8")
    print(f"✅ Report saved to: {report_file}")
    return str(report_file)

# Generate report
if not plans_df.empty:
    report_path = generate_performance_report(runs_df, plans_df, types_df)
    print(f"\n📄 Full report available at: {report_path}")
else:
    print("⚠️  No data available to generate report.")


In [ ]:
# Export Data to CSV
if not plans_df.empty:
    export_dir = Path("runtime/outputs/performance_reports")
    export_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Export plans data
    plans_export = plans_df.copy()
    plans_export['timestamp'] = plans_export['timestamp'].astype(str)
    plans_file = export_dir / f"plans_data_{timestamp}.csv"
    plans_export.to_csv(plans_file, index=False)
    print(f"✅ Plans data exported to: {plans_file}")
    
    # Export types data
    if not types_df.empty:
        types_export = types_df.copy()
        types_export['timestamp'] = types_export['timestamp'].astype(str)
        types_file = export_dir / f"types_data_{timestamp}.csv"
        types_export.to_csv(types_file, index=False)
        print(f"✅ Types data exported to: {types_file}")
    
    # Export runs data
    if not runs_df.empty:
        runs_export = runs_df.copy()
        runs_export['timestamp'] = runs_export['timestamp'].astype(str)
        runs_file = export_dir / f"runs_data_{timestamp}.csv"
        runs_export.to_csv(runs_file, index=False)
        print(f"✅ Runs data exported to: {runs_file}")
else:
    print("⚠️  No data available to export.")


In [ ]:
# Save Charts as Images
if not plans_df.empty:
    export_dir = Path("runtime/outputs/performance_reports")
    export_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save confidence trends chart
    if len(plans_df) > 0:
        fig, ax = plt.subplots(figsize=(14, 8))
        for plan_name in sorted(plans_df['plan_name'].unique()):
            plan_data = plans_df[plans_df['plan_name'] == plan_name].sort_values('timestamp')
            if not plan_data.empty:
                ax.plot(plan_data['timestamp'], plan_data['average_confidence'], 
                       marker='o', label=plan_name, linewidth=2, markersize=6, alpha=0.7)
        ax.axhline(y=0.8, color='r', linestyle='--', linewidth=2, label='Target (0.8)', alpha=0.7)
        ax.set_title('Confidence Score Trends Over Time', fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Timestamp', fontsize=12)
        ax.set_ylabel('Average Confidence', fontsize=12)
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        chart_file = export_dir / f"confidence_trends_{timestamp}.png"
        plt.savefig(chart_file, dpi=300, bbox_inches='tight')
        print(f"✅ Chart saved to: {chart_file}")
        plt.close()
    
    print(f"\n💡 Tip: All exports saved to {export_dir}")
else:
    print("⚠️  No data available to export charts.")
